### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import TrainingArguments
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import SorlTrainer

device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
    memory_span=1792
)
model = model.to(device)

print(f"Model loaded: {model_name}")
print(f"Base vocab size: {model.vocab_sizes[0].item()}")
print(f"Abstract vocab size: {model.total_vocab_size - model.vocab_sizes[0]}")
print(f"Abstract token range: [{model.vocab_sizes[0].item()}, {model.total_vocab_size})")

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded: Qwen/Qwen2.5-0.5B
Base vocab size: 151936
Abstract vocab size: 129
Abstract token range: [151936, 152065)


In [3]:
input_ids = torch.randint(0, model.vocab_sizes[0].item(), (1, 4)).to(device)

# --- generate (with periodic abstract tokens) ---
seq_with_abs = model.generate(input_ids, max_new_tokens=8, K=3)

# --- recursion --- 
seq_recursed, _ = model.recursion(seq_with_abs, max_iterations=2, memory_span_abs=1792, memory_span_traj=1792, attn_blocksize=1792, temperature=0.0)

# --- rollout 'n' p(a | s), then pick the best rollout with p(s | a) ---
from sorl.sorl_trainer import sorl_search
search_ids, search_ppt, search_adv = sorl_search(model, input_ids, n=8)

In [4]:
# --- Training with SoRL Trainer ---

# Create dummy training data
train_data = torch.randint(0, model.vocab_sizes[0].item(), (10, 16)).to(device)
train_labels = train_data.clone()

# Create a simple dataset
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return {"input_ids": self.data[idx]}

train_dataset = SimpleDataset(train_data)

# Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sorl_results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    warmup_steps=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=1,
    save_steps=10,
    eval_steps=10,
)

# Create SoRL trainer
print("Creating SoRL trainer...")
trainer = SorlTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    num_rollouts=4,
    K=3,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_info_gain=10.0,
    alpha_abs=0.1,
    alpha_soft_zipf=1.0,
)

print("Trainer created successfully!")
print(f"Model vocab sizes: {model.vocab_sizes}")
print(f"Total vocab size: {model.total_vocab_size}")

# Test a single training step
print("\n=== Testing Training Step ===")
sample_batch = {"input_ids": train_data[:2]}  # Take first 2 samples
print(f"Batch shape: {sample_batch['input_ids'].shape}")

# Compute loss (this will use SoRL search internally)
loss = trainer.compute_loss(model, sample_batch)
print(f"Training loss: {loss.item():.4f}")

print("\n✅ SoRL training pipeline is working!")

Creating SoRL trainer...
Trainer created successfully!
Model vocab sizes: tensor([151936,    129])
Total vocab size: 152065

=== Testing Training Step ===
Batch shape: torch.Size([2, 16])


/Users/ksgk/Implementation/mod_gpt/sorl/sorl_trainer.py:187: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SorlTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


Training loss: 1054.8452

✅ SoRL training pipeline is working!


In [5]:
trainer.train()

wandb: Currently logged in as: fangyuan-yu18 (ksgk-hack) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,905.208500
2,756.570100
3,590.757100
4,488.288400
5,393.819200


TrainOutput(global_step=5, training_loss=626.9286499023438, metrics={'train_runtime': 14.1827, 'train_samples_per_second': 0.705, 'train_steps_per_second': 0.353, 'total_flos': 474382417920.0, 'train_loss': 626.9286499023438, 'epoch': 1.0})